# AI251P Applied Computer Vision — Project 9
## Phase 1: Waste Material Classification
**Dataset:** TrashNet (six material classes)
**Next stage:** detection and instance segmentation on ZeroWaste-f


### How we run the experiments
1. Select a GPU runtime in Google Colab.
2. Run the notebook in order. All results are temporarily saved in the Colab runtime under /content/KAUST_Project9_Phase1. At the final cell, download the ZIP to your computer. Do not close or restart Colab before downloading results.
3. Review 60 sampled labels manually, update `label_audit_review.csv`, then rerun the audit summary.
4. Check all three model runs and synchronize W&B results before compiling the report.

**Important:** Results are only valid after running the cells on the real dataset. The dataset is not included in the submission.


### حفظ النتائج بدون Google Drive
هذا الإصدار لا يطلب صلاحية Google Drive. يتم حفظ النتائج داخل جلسة Colab مؤقتًا، وفي الخلية الأخيرة تُنزّل نسخة ZIP إلى جهازك. **مهم:** إذا أغلقت الجلسة أو انتهت قبل التنزيل قد تفقد النتائج. بعد إنشاء `label_audit_review.csv` يمكنك مراجعته يدويًا عبر لوحة Files في Colab، ثم إعادة تشغيل خلية مراجعة التسميات.


In [ ]:
%pip -q install "huggingface_hub>=0.26,<2" "ImageHash>=4.3,<5" "wandb>=0.18,<1" "scikit-learn>=1.4,<2" "seaborn>=0.13,<1" "tqdm>=4.66,<5" "tabulate>=0.9,<1"

## 1. Setup
We fix a random seed and record the main settings to make the runs easier to reproduce.


In [ ]:
import os, sys, json, random, time, hashlib, zipfile, shutil, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm
from IPython.display import display
import imagehash, wandb, torch, torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report, confusion_matrix, top_k_accuracy_score
from scipy.stats import binomtest, beta
from huggingface_hub import hf_hub_download
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
# If performance suffers on T4, deterministic algorithms can be relaxed and documented.
torch.backends.cudnn.benchmark=False
USE_DRIVE=False  # Privacy-first: save outputs locally in Colab; download ZIP before closing session
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/KAUST_Project9_Phase1')
else:
    OUT=Path('/content/KAUST_Project9_Phase1')
OUT.mkdir(parents=True, exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
(OUT/'figures').mkdir(exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE=160
BATCH_SIZE=32 if torch.cuda.is_available() else 16
NUM_WORKERS=2
EPOCHS={'scratch':6,'frozen':4,'finetune':4}  # Short emergency baseline, tune ONLY on validation
# If you have a W&B account, get your API key via Colab's Secrets panel:
# key name WANDB_API_KEY, grant notebook access. Never hardcode account tokens.
if 'google.colab' in sys.modules:
    try:
        from google.colab import userdata
        if not os.getenv('WANDB_API_KEY'):
            secret=userdata.get('WANDB_API_KEY')
            if secret: os.environ['WANDB_API_KEY']=secret
    except Exception: pass
WANDB_MODE='online' if os.getenv('WANDB_API_KEY') else 'offline'
print('Device:',DEVICE,'Outputs:',OUT,'W&B:',WANDB_MODE)
print('PyTorch:',torch.__version__,'torchvision:',torchvision.__version__)
(OUT/'config.json').write_text(json.dumps(dict(seed=SEED, image_size=IMG_SIZE,batch_size=BATCH_SIZE,epochs=EPOCHS,device=str(DEVICE),wandb_mode=WANDB_MODE),indent=2))

## 2. Download and inspect TrashNet
We use the resized TrashNet images, check for damaged files, and identify exact duplicates before splitting the data.


In [ ]:
zip_path=Path('/content/dataset-resized.zip')
if not zip_path.exists():
    try:
        zip_path=Path(hf_hub_download(repo_id='garythung/trashnet', repo_type='dataset', filename='dataset-resized.zip'))
    except Exception as e:
        try:
            from urllib.request import urlretrieve
            fallback='https://github.com/garythung/trashnet/raw/refs/heads/master/data/dataset-resized.zip'
            zip_path=Path('/content/dataset-resized.zip')
            urlretrieve(fallback,zip_path)
        except Exception as e2:
            raise RuntimeError('Could not fetch resized TrashNet. Download dataset-resized.zip manually from https://github.com/garythung/trashnet/tree/master/data and upload to /content/.') from e2
if not zipfile.is_zipfile(zip_path):
    raise RuntimeError('Dataset download is not a valid zip. Upload the correct dataset-resized.zip manually.')
EXTRACT=Path('/content/KAUST_TrashNet_Extracted')
EXTRACT.mkdir(exist_ok=True)
if not list(EXTRACT.rglob('*.jpg')):
    with zipfile.ZipFile(zip_path) as z:
        # Reject unexpected traversal paths from any downloaded archive.
        for info in z.infolist():
            dest=(EXTRACT/info.filename).resolve()
            if not str(dest).startswith(str(EXTRACT.resolve())+os.sep):
                raise ValueError('Unsafe archive member: '+info.filename)
        z.extractall(EXTRACT)
CLASSES=['cardboard','glass','metal','paper','plastic','trash']
paths=sorted(p for p in EXTRACT.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png') and p.parent.name.lower() in CLASSES and '__MACOSX' not in p.parts)
assert len(paths)>2000, f'Expected about 2527 JPG images, found {len(paths)}. Inspect zip structure.'
print('Images found:',len(paths)); print('First:',str(paths[0]))

In [ ]:
records=[]; bad=[]; exact_first={}; exact_repeats=[]
for p in tqdm(paths,desc='Verify/hash images'):
    label=p.parent.name.lower()
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im:
            rgb=ImageOps.exif_transpose(im).convert('RGB')
            ph=str(imagehash.phash(rgb))
            width,height=rgb.size
        digest=hashlib.sha256(p.read_bytes()).hexdigest()
        if digest in exact_first:
            exact_repeats.append((str(p),exact_first[digest][0],label,exact_first[digest][1]))
            continue
        exact_first[digest]=(str(p),label)
        records.append(dict(path=str(p),label_name=label,phash=ph,sha256=digest,width=width,height=height))
    except (UnidentifiedImageError,OSError,ValueError) as e:
        bad.append((str(p),repr(e)))
df=pd.DataFrame(records)
assert len(df)>2000, 'Too few valid images; inspect the downloaded archive'
pd.DataFrame(exact_repeats,columns=['removed_path','kept_path','removed_label','kept_label']).to_csv(OUT/'exact_duplicates_removed.csv',index=False)
pd.DataFrame(bad,columns=['path','error']).to_csv(OUT/'corrupt_images.csv',index=False)
print('Valid unique images:',len(df),'Exact duplicates removed:',len(exact_repeats),'Invalid:',len(bad))
print(df.label_name.value_counts().reindex(CLASSES).to_string())

## 3. Prepare train, validation, and test splits
Near-duplicate photographs can create misleading test scores. We group images with similar perceptual hashes so the same group does not appear in two different splits. This reduces leakage risk, although capture-session metadata are unavailable.


In [ ]:
n=len(df); parent=list(range(n)); sizes=[1]*n
def find(i):
    while parent[i]!=i:
        parent[i]=parent[parent[i]];i=parent[i]
    return i
def union(i,j):
    a,b=find(i),find(j)
    if a==b:return
    if sizes[a]<sizes[b]:a,b=b,a
    parent[b]=a;sizes[a]+=sizes[b]
hashes=[int(v,16) for v in df.phash]
for i in tqdm(range(n),desc='Near-duplicate grouping'):
    for j in range(i):
        if (hashes[i]^hashes[j]).bit_count()<=2:
            union(i,j)
df['group_id']=[find(i) for i in range(n)]
print('Groups:',df.group_id.nunique(),'Largest:',df.group_id.value_counts().max())
sg1=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
trainval_idx,test_idx=next(sg1.split(df,df.label_name,df.group_id))
trainval=df.iloc[trainval_idx]
sg2=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=SEED+1)
tr_rel,val_rel=next(sg2.split(trainval,trainval.label_name,trainval.group_id))
df['split']='test'
df.loc[trainval.index[tr_rel],'split']='train'
df.loc[trainval.index[val_rel],'split']='val'
a=[set(df.loc[df.split==s,'group_id']) for s in ('train','val','test')]
assert all(a[i].isdisjoint(a[j]) for i in range(3) for j in range(i+1,3)), 'Group leakage!'
assert all(df.loc[df.split==s,'label_name'].nunique()==len(CLASSES) for s in ('train','val','test')), 'Missing a class in split'
ct=pd.crosstab(df.label_name,df.split).reindex(index=CLASSES,columns=['train','val','test'],fill_value=0)
print('Split count:',df.split.value_counts().to_dict());display(ct)
df.to_csv(OUT/'split_manifest.csv',index=False);ct.to_csv(OUT/'class_counts.csv')
ct.plot(kind='bar',figsize=(9,4));plt.ylabel('Images');plt.tight_layout();plt.savefig(OUT/'figures/class_balance.png',dpi=150);plt.show()

## 4. Manual label check
We sample 10 images from each of the six classes. A team member reviews these pictures and records which labels are correct. The code must not fill in the human review on its own.


In [ ]:
audit_rows=[]
for c in CLASSES:
    audit_rows.extend(df[df.label_name==c].sample(n=10,random_state=SEED).to_dict('records'))
audit=pd.DataFrame(audit_rows)
audit.insert(0,'audit_id',range(1,len(audit)+1))
audit['correct']='';audit['reviewer_label']='';audit['notes']=''
review_csv=OUT/'label_audit_review.csv'
# Preserve real HUMAN auditing if notebook is rerun after initial review.
if review_csv.exists():
    previous=pd.read_csv(review_csv,dtype={'correct':'string'}).fillna('')
    previous=previous.drop_duplicates('path',keep='last').set_index('path')
    for field in ('correct','reviewer_label','notes'):
        audit[field]=audit['path'].map(previous[field]).fillna('') if field in previous.columns else ''
audit[['audit_id','path','label_name','correct','reviewer_label','notes']].to_csv(review_csv,index=False)
for page in range(2):
    sub=audit.iloc[page*30:(page+1)*30]
    fig,axes=plt.subplots(5,6,figsize=(18,13))
    for ax in axes.flat:ax.axis('off')
    for ax,(_,r) in zip(axes.flat,sub.iterrows()):
        with Image.open(r['path']) as im:ax.imshow(im.convert('RGB'))
        ax.set_title(f"#{r['audit_id']} {r['label_name']}",fontsize=9)
    fig.tight_layout();fig.savefig(OUT/f'figures/label_audit_page_{page+1}.png',dpi=160);plt.show()
print('Human review required: edit',OUT/'label_audit_review.csv')

In [ ]:
# Rerun AFTER filling the correct column in label_audit_review.csv
review=pd.read_csv(OUT/'label_audit_review.csv')
valid=review['correct'].astype('string').str.strip().isin(['0','1'])
reviewed=review.loc[valid].copy()
if len(reviewed)==0:
    audit_text='PENDING: No human label review recorded. Do not claim an audited error rate.'
else:
    errors=int((reviewed['correct'].astype(int)==0).sum()); k=len(reviewed)
    upper=float(beta.ppf(0.95,errors+1,k-errors)) if errors<k else 1.0
    audit_text=f'Human-reviewed n={k}, observed errors={errors}; one-sided exact 95% upper error-rate bound={upper:.4f}'
(OUT/'label_audit_result.txt').write_text(audit_text);print(audit_text)

## 5. Image preprocessing
Training images use small random transformations. Validation and test images use fixed resizing and normalization.


In [ ]:
class_to_id={x:i for i,x in enumerate(CLASSES)}
df['label_id']=df.label_name.map(class_to_id).astype(int)
df.to_csv(OUT/'split_manifest.csv',index=False)
train_tf=transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE,scale=(0.85,1.0)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(brightness=.1,contrast=.1),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
val_tf=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class ImageDS(Dataset):
    def __init__(self,frame,tf):self.frame=frame.reset_index(drop=True);self.tf=tf
    def __len__(self):return len(self.frame)
    def __getitem__(self,idx):
        r=self.frame.iloc[idx]
        with Image.open(r['path']) as im:x=self.tf(im.convert('RGB'))
        return x,int(r['label_id'])
frames={s:df[df.split==s].copy() for s in ('train','val','test')}
loaders={s:DataLoader(ImageDS(frames[s],train_tf if s=='train' else val_tf),batch_size=BATCH_SIZE,shuffle=(s=='train'),num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available()) for s in frames}
counts=frames['train'].label_id.value_counts().reindex(range(len(CLASSES)),fill_value=1)
class_weights=torch.tensor(len(frames['train'])/(len(CLASSES)*counts.values),dtype=torch.float32,device=DEVICE)
print('Class weights:',dict(zip(CLASSES,class_weights.cpu().tolist())))

## 6. Compare classification models
We compare (1) a CNN trained from scratch, (2) a frozen ResNet18 backbone, and (3) full ResNet18 fine-tuning. The best run is selected by validation macro-F1, not the test results.


In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        layers=[];prev=3
        for ch in [32,64,128,192]:
            layers += [nn.Conv2d(prev,ch,3,padding=1),nn.BatchNorm2d(ch),nn.ReLU(inplace=True),nn.MaxPool2d(2)]
            prev=ch
        self.features=nn.Sequential(*layers)
        self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.35),nn.Linear(prev,num_classes))
    def forward(self,x):return self.head(self.features(x))
def make_model(kind):
    if kind=='scratch':return ScratchCNN(len(CLASSES)).to(DEVICE)
    m=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc=nn.Linear(m.fc.in_features,len(CLASSES))
    if kind=='frozen':
        for p in m.parameters():p.requires_grad=False
        for p in m.fc.parameters():p.requires_grad=True
    return m.to(DEVICE)
@torch.no_grad()
def predict(model,loader):
    model.eval();ys=[];probs=[]
    for x,y in loader:
        p=torch.softmax(model(x.to(DEVICE)),dim=1).cpu().numpy()
        ys.extend(y.numpy().tolist());probs.append(p)
    return np.asarray(ys,dtype=int),np.concatenate(probs,axis=0)
def ece(y,p,bins=10):
    conf=p.max(1);pred=p.argmax(1);value=0.0
    for low,high in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        idx=(conf>=low)&((conf<=high) if high==1 else (conf<high))
        if idx.any():value+=float(idx.mean())*abs(float((pred[idx]==y[idx]).mean())-float(conf[idx].mean()))
    return value
def quick_metrics(y,p):
    pred=p.argmax(1)
    return {'accuracy':float(accuracy_score(y,pred)),'macro_f1':float(f1_score(y,pred,labels=list(range(len(CLASSES))),average='macro',zero_division=0)),'ece':float(ece(y,p))}
def run_train(kind):
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    model=make_model(kind)
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-3 if kind!='finetune' else 1e-4,weight_decay=1e-4)
    lossfn=nn.CrossEntropyLoss(weight=class_weights)
    run=wandb.init(project='KAUST-Waste-Project9',name=f'P1-{kind}-seed{SEED}',config={'model':kind,'seed':SEED,'img_size':IMG_SIZE,'epochs':EPOCHS[kind],'batch_size':BATCH_SIZE,'split':'stratified_group_phash','classes':CLASSES},mode=WANDB_MODE,dir=str(OUT),reinit=True)
    history=[];best=-1;bestpath=OUT/'checkpoints'/f'{kind}_best.pt'
    for epoch in range(EPOCHS[kind]):
        model.train();cum=0.0;seen=0
        if kind=='frozen':
            model.eval()  # Keep pretrained BatchNorm running statistics frozen
            model.fc.train()
        for x,y in tqdm(loaders['train'],desc=f'{kind} epoch {epoch+1}/{EPOCHS[kind]}',leave=False):
            x=x.to(DEVICE);y=y.to(DEVICE);opt.zero_grad(set_to_none=True)
            logits=model(x);loss=lossfn(logits,y);loss.backward();opt.step()
            cum+=float(loss.item())*len(y);seen+=len(y)
        yv,pv=predict(model,loaders['val']);m=quick_metrics(yv,pv)
        item={'epoch':epoch+1,'train_loss':cum/max(1,seen),'val_accuracy':m['accuracy'],'val_macro_f1':m['macro_f1'],'val_ece':m['ece']}
        history.append(item);wandb.log(item)
        print(kind,epoch+1,'val F1',round(m['macro_f1'],4),'val accuracy',round(m['accuracy'],4))
        if m['macro_f1']>best:
            best=m['macro_f1']
            torch.save({'state_dict':model.state_dict(),'kind':kind,'classes':CLASSES,'img_size':IMG_SIZE,'seed':SEED},bestpath)
    wandb.summary['best_val_macro_f1']=best
    wandb.finish()
    pd.DataFrame(history).to_csv(OUT/f'history_{kind}.csv',index=False)
    return {'kind':kind,'best_val_macro_f1':float(best),'checkpoint':str(bestpath)}
# Save progress after each experiment; rerun will skip completed experiments.
results=[]
for kind in ['scratch','frozen','finetune']:
    summary_file=OUT/f'completed_{kind}.json'
    if summary_file.exists() and (OUT/'checkpoints'/f'{kind}_best.pt').exists():
        record=json.loads(summary_file.read_text())
        print('Already completed:',kind,'validation macro F1',record['best_val_macro_f1'])
    else:
        record=run_train(kind)
        summary_file.write_text(json.dumps(record,indent=2))
    results.append(record)
results_df=pd.DataFrame(results).sort_values('best_val_macro_f1',ascending=False)
results_df.to_csv(OUT/'val_model_comparison.csv',index=False)
display(results_df)
fig,ax=plt.subplots(figsize=(8,4))
for kind in ['scratch','frozen','finetune']:
    h=pd.read_csv(OUT/f'history_{kind}.csv');ax.plot(h['epoch'],h['val_macro_f1'],marker='o',label=kind)
ax.legend();ax.set(xlabel='Epoch',ylabel='Validation Macro-F1',title='Model comparison (validation only)');fig.tight_layout();fig.savefig(OUT/'figures/learning_curves.png',dpi=160);plt.show()

## 7. Final evaluation
After choosing the best model on validation data, we evaluate it once on the held-out test set. We save class-level metrics, a confusion matrix, and some mistakes for discussion.


In [ ]:
best_kind=results_df.iloc[0]['kind']
print('Held-out final test: run only once after selection and keep outputs. Selected via validation:',best_kind)
ckpt=torch.load(OUT/'checkpoints'/f'{best_kind}_best.pt',map_location='cpu',weights_only=False)
model=make_model(best_kind);model.load_state_dict(ckpt['state_dict']);model.eval()
yt,pt=predict(model,loaders['test']);pred=pt.argmax(1)
m=quick_metrics(yt,pt)
m['top3_accuracy']=float(top_k_accuracy_score(yt,pt,k=3,labels=list(range(len(CLASSES)))))
correct=int((yt==pred).sum());ci=binomtest(correct,len(yt)).proportion_ci(confidence_level=.95,method='exact')
m.update(test_n=len(yt),accuracy_ci95_low=float(ci.low),accuracy_ci95_high=float(ci.high),selected_model=best_kind)
(OUT/'test_metrics.json').write_text(json.dumps(m,ensure_ascii=False,indent=2));print(json.dumps(m,indent=2))
report=classification_report(yt,pred,labels=list(range(len(CLASSES))),target_names=CLASSES,zero_division=0,output_dict=True)
pd.DataFrame(report).transpose().to_csv(OUT/'test_classification_report.csv')
cm=confusion_matrix(yt,pred,labels=list(range(len(CLASSES))))
fig,ax=plt.subplots(figsize=(8,6));sns.heatmap(cm,annot=True,fmt='d',xticklabels=CLASSES,yticklabels=CLASSES,ax=ax)
ax.set(xlabel='Predicted',ylabel='Ground truth',title=f'Test confusion matrix — {best_kind}')
fig.tight_layout();fig.savefig(OUT/'figures/confusion_matrix_test.png',dpi=170);plt.show()
err_idx=np.where(yt!=pred)[0]
errors=frames['test'].iloc[err_idx].copy();errors['predicted_label']=[CLASSES[v] for v in pred[err_idx]];errors['confidence']=pt[err_idx].max(1)
errors.to_csv(OUT/'test_mistakes.csv',index=False)
if len(errors):
    example=errors.sort_values('confidence',ascending=False).head(12)
    fig,axes=plt.subplots(3,4,figsize=(13,10))
    for ax in axes.flat:ax.axis('off')
    for ax,(_,r) in zip(axes.flat,example.iterrows()):
        with Image.open(r['path']) as im:ax.imshow(im.convert('RGB'))
        ax.set_title(f"True {r['label_name']} | Pred {r['predicted_label']}\nconf {r['confidence']:.2f}",fontsize=9)
    fig.tight_layout();fig.savefig(OUT/'figures/error_gallery.png',dpi=150);plt.show()
run=wandb.init(project='KAUST-Waste-Project9',name=f'P1-FINAL-TEST-{best_kind}',mode=WANDB_MODE,dir=str(OUT),reinit=True)
wandb.log({'test_accuracy':m['accuracy'],'test_macro_f1':m['macro_f1'],'test_ece':m['ece'],'test_top3_accuracy':m['top3_accuracy'],'test_n':m['test_n']})
run.finish()

## 8. Testing on conveyor-belt images
TrashNet pictures are much simpler than factory conveyor-belt scenes. If labelled ZeroWaste-f crops are available, we test how the model performs on this different type of image. If they are missing, the notebook records the limitation instead of guessing results.


In [ ]:
external_root=Path('/content/zerowaste_crops')
external_paths=[p for p in external_root.rglob('*') if p.suffix.lower() in ['.png','.jpg','.jpeg'] and p.parent.name.lower() in ['cardboard','plastic','metal']] if external_root.exists() else []
if not external_paths:
    print('PENDING: ZeroWaste-f annotated crops unavailable. Assign preparation of labelled crops to Team Member 3 NOW. Cross-domain scores cannot be stated.')
    (OUT/'zerowaste_cross_domain_status.txt').write_text('PENDING: annotated ZeroWaste-f crop dataset not supplied')
else:
    ext=pd.DataFrame([{'path':str(p),'label_name':p.parent.name.lower(),'label_id':class_to_id[p.parent.name.lower()]} for p in external_paths])
    extloader=DataLoader(ImageDS(ext,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS)
    yy,pp=predict(model,extloader);yp=pp.argmax(1);overlap=[class_to_id[c] for c in ['cardboard','plastic','metal']]
    stats={'n':len(yy),'accuracy':float(accuracy_score(yy,yp)),'macro_f1_overlap_classes':float(f1_score(yy,yp,labels=overlap,average='macro',zero_division=0)),'note':'TrashNet six-class predictions evaluated on mapped ZeroWaste-f material crops'}
    print(stats);(OUT/'zerowaste_cross_domain_results.json').write_text(json.dumps(stats,indent=2))

## 9. Reproducibility and outputs
Keep the configuration, split manifest, experiment logs, evaluation plots, and model checkpoint together. Raw training images are not part of the code repository.


## 10. Report generated from actual results
The next cell creates a technical summary from the files generated during execution. Check the text carefully and add your team's own error analysis before submitting.


In [ ]:
#!/usr/bin/env python3
"""Create an evidence-backed Phase 1 English report from real Colab output files.
Usage: python generate_report.py /content/KAUST_Project9_Phase1
Does NOT invent experimental results; lists outstanding work explicitly.
"""
from __future__ import annotations
import sys,json,csv,datetime,math
from pathlib import Path

CATS=['cardboard','glass','metal','paper','plastic','trash']

def safe_json(p):
    try: return json.loads(p.read_text(encoding='utf-8'))
    except (OSError,ValueError): return None

def csv_rows(p):
    try:
        with p.open(newline='',encoding='utf-8-sig') as f: return list(csv.DictReader(f))
    except (OSError,UnicodeError):return []

def val(d,key,fmt=None):
    v=(d or {}).get(key,None)
    if v is None or v=='':return '**PENDING: requires actual experiment**'
    if isinstance(v,float) and fmt:return format(v,fmt)
    return str(v)

def fnum(v,fmt='.4f'):
    try:return format(float(v),fmt)
    except (ValueError,TypeError):return '**PENDING**'

def main(folder):
    root=Path(folder)
    root.mkdir(parents=True,exist_ok=True)
    met=safe_json(root/'test_metrics.json')
    conf=safe_json(root/'config.json')
    compare=csv_rows(root/'val_model_comparison.csv')
    class_counts=csv_rows(root/'class_counts.csv')
    classes=csv_rows(root/'test_classification_report.csv')
    problems=csv_rows(root/'test_mistakes.csv')
    auditpath=root/'label_audit_review.csv'
    audit=csv_rows(auditpath)
    reviews=[r for r in audit if str(r.get('correct','')).strip() in ('0','1')]
    from collections import Counter
    counts=Counter(r.get('label_name') for r in reviews)
    err_count=sum(str(r.get('correct','')).strip()=='0' for r in reviews)
    if reviews:
        from scipy.stats import beta
        n=len(reviews)
        b=float(beta.ppf(.95,err_count+1,n-err_count)) if err_count<n else 1.
        audit_txt=f'{n} of {len(audit)} sampled images reviewed; {err_count} apparent label errors; exact one-sided 95% upper bound = {b:.4f}. Per-class review counts: {dict(counts)}.'
    else: audit_txt='**PENDING: team member must visually review the 60 sampled images; do not claim an audit.**'
    if class_counts:
        count_table='| Class | Train | Val | Test |\n|---|---:|---:|---:|\n'
        for r in class_counts:
            label=r.get('label_name') or r.get('Unnamed: 0') or next(iter(r.values()))
            count_table+=f"| {label} | {r.get('train','?')} | {r.get('val','?')} | {r.get('test','?')} |\n"
    else: count_table='**PENDING: run data cleaning and split cells to generate class_counts.csv.**'
    if compare:
        cmp_table='| Training run | Best validation macro-F1 | Checkpoint |\n|---|---:|---|\n'
        for r in compare:
            cmp_table+=f"| {r.get('kind','?')} | {fnum(r.get('best_val_macro_f1'))} | `{Path(r.get('checkpoint','?')).name}` |\n"
    else: cmp_table='**PENDING: run ScratchCNN, frozen ResNet18 and fine-tuned ResNet18; include all W&B runs.**'
    if classes:
        ctable='| Class | Precision | Recall | F1 | Support |\n|---|---:|---:|---:|---:|\n'
        for r in classes:
            name=r.get('Unnamed: 0',r.get('label',''))
            if name in CATS:
                ctable+=f"| {name} | {fnum(r.get('precision'))} | {fnum(r.get('recall'))} | {fnum(r.get('f1-score'))} | {r.get('support','?')} |\n"
    else: ctable='**PENDING: generate test_classification_report.csv.**'
    errtable='| Actual | Predicted | Confidence | Image |\n|---|---|---:|---|\n'
    if problems:
        for r in sorted(problems,key=lambda x:float(x.get('confidence',0)),reverse=True)[:5]:
            errtable+=f"| {r.get('label_name','?')} | {r.get('predicted_label','?')} | {fnum(r.get('confidence'))} | `{Path(r.get('path','?')).name}` |\n"
    else:
        errtable+='| PENDING | PENDING | — | Examine held-out errors after model runs |\n'
    domain=safe_json(root/'zerowaste_cross_domain_results.json')
    domaintext=(f"Evaluation on {domain.get('n','?')} annotated conveyor crops yielded macro-F1 on overlapping mapped classes = {fnum(domain.get('macro_f1_overlap_classes'))}; overall accuracy = {fnum(domain.get('accuracy'))}." if domain else '**PENDING: ZeroWaste-f cropped, annotated and held-out samples were not provided or evaluated.** This is a required subject-specific part of the example topic, so flag it for completion.')
    hasmodel=(root/'best_model.pt').exists()
    check={
      'Verified data and grouped split':(root/'split_manifest.csv').exists() and (root/'class_counts.csv').exists(),
      'Manual label audit complete':len(reviews)==len(audit) and len(audit)>=60,
      'Scratch baseline and two transfer-learning runs':len(compare)>=3,
      'Real held-out test metrics':bool(met),
      'Evaluation plots':all((root/'figures'/f).exists() for f in ['class_balance.png','learning_curves.png','confusion_matrix_test.png']),
      'Best model saved':hasmodel,
      'ZeroWaste-f cross-domain assessment':bool(domain),
      'W&B runs made available online':False,
      'Independent rerun by team member':False,
      'Instructor approves 5-person advanced work':False,
    }
    checks='\n'.join(f"- [{'x' if ok else ' '}] {n}" for n,ok in check.items())
    exp_size=val(conf,'image_size')
    best=val(met,'selected_model')
    content=f'''# AI251P Applied Computer Vision — Capstone Project 9
## Phase 1 report (M1–M3): Waste Recognition and Segmentation on Conveyor Belts
**KAUST Academy — Academic project — generated {datetime.date.today().isoformat()}**
**Due date:** 28 October 2026. **Weight:** 40/100 points.
**Team:** [INSERT FIVE STUDENT NAMES AND ACTUAL CONTRIBUTIONS]

> **Evidence policy.** This document is generated from actual Colab output files. PENDING means the corresponding experiment or human check has NOT been completed. Never submit a PENDING item as a measured result. This is the Phase 1 classification component, NOT a completed end-to-end recycling system.

### 1. Aim and task scope
We aim to recognise waste materials as one component of a recycling-belt vision system. A reproducible six-class image classifier is developed on TrashNet. Phase 2 will detect objects in ZeroWaste-f conveyor images, segment each instance, derive pick points and benchmark an integrated service. The project also assesses the distribution shift between clean single-object photographs and cluttered conveyor images.

### 2. Data sources, provenance, and limitations (M1)
The project guidelines specify three publicly described datasets: TrashNet for classification, ZeroWaste-f for detection and segmentation, and TACO as an external domain dataset. Phase 1 classifier images come from Thung and Yang's TrashNet (2,527 source images, six categories). The project uses the *resized archive*, not the 5,054-row Hugging Face viewer that may combine original and resized images. Its six classes are cardboard, glass, metal, paper, plastic, and general trash. Source: https://github.com/garythung/trashnet ; hosting: https://huggingface.co/datasets/garythung/trashnet . The hosting dataset page lists MIT. Raw images are excluded from submission code repositories and are subject to their respective licences.

The code verifies readable images, records SHA-256 exact duplicate hashes and perceptual pHashes, removes exact duplicate binaries and groups near-duplicate images (Hamming distance at most 2) before a stratified group split. This reduces some forms of leakage but cannot guarantee session-level independence because source sessions/cameras are not identified. Each group belongs to one split; held-out test data are reserved for one final evaluation, not model selection. Image sizes, transforms, exact class balances and split identifiers are preserved in output files.

**Observed counts by split** (never infer totals from the original paper):

{count_table}

**Label audit:** {audit_txt}

**Limitations:** TrashNet mostly contains isolated objects against a plain background, while ZeroWaste-f comprises conveyor-belt scenes with object overlap, deformation, and different material labels. Public footage may have temporal similarity; future object detection splits must be separated by video sequence. A pHash/group split in TrashNet does not remove the domain shift. Minority classes require macro-F1 and per-class metrics, not overall accuracy alone.

### 3. Method — from-scratch CNN (M2)
The ScratchCNN baseline uses four Conv2D + BatchNorm + ReLU + pooling blocks (channel widths 32/64/128/192), global-average pooling and a dropout linear head. Training applies random resized crops, random flips and mild colour jitter only to the training split. Validation/test transform resizes to **{exp_size} x {exp_size}** and uses ImageNet channel normalization (same policy across comparisons). Cross-entropy class weights derive from the *training* class counts; optimizer: AdamW, weight decay 1e-4; seed {val(conf,'seed')}.

The notebook saves the best checkpoint on *validation macro-F1*, training/validation curves, configuration, class weights, and error examples. Test data are not used for tuning.

### 4. Transfer learning (M3) and model comparison
ResNet18 pretrained on ImageNet is compared in two settings: (1) frozen backbone and trained classification head, and (2) full backbone fine-tuning. The same split and evaluation metric are used for all three experiments. The code tracks loss, accuracy, macro-F1 and calibration error by epoch in Weights & Biases. URLs need to be added after runs are synchronised online; offline-only runs are insufficient evidence of an online W&B experiment dashboard.

{cmp_table}

**W&B project/run URLs:** [TEAM MUST FILL AFTER REAL ONLINE LOGGING].

### 5. Held-out test results (only after model selection)
Model selected by validation macro-F1: **{best}**. Input resolution: {exp_size}x{exp_size}. Metrics below are from `test_metrics.json` when present.

| Metric | Observed value |
|---|---:|
| Test image count | {val(met,'test_n')} |
| Top-1 accuracy | {fnum((met or {}).get('accuracy'))} |
| Top-3 accuracy | {fnum((met or {}).get('top3_accuracy'))} |
| Macro F1 | {fnum((met or {}).get('macro_f1'))} |
| Expected calibration error, 10 bins | {fnum((met or {}).get('ece'))} |
| Exact 95% accuracy confidence interval | [{fnum((met or {}).get('accuracy_ci95_low'))}, {fnum((met or {}).get('accuracy_ci95_high'))}] |

**Per-class measures:**

{ctable}

**Error investigation (examples, not inferred causes):**

{errtable}

Interpretation of false predictions, with visual evidence: [TEAM MUST REVIEW ACTUAL IMAGE GALLERY AND WRITE AT LEAST THREE CAUSES: background, reflective material, deformation, packaging appearance, etc. If fewer than 3 errors, discuss the available cases.]

Figures: `figures/class_balance.png`, `figures/learning_curves.png`, `figures/confusion_matrix_test.png`, `figures/error_gallery.png` (when the model makes mistakes).

### 6. Cross-domain test: ZeroWaste-f
{domaintext}

Important label mapping: ZeroWaste-f includes categories such as soft and rigid plastic; the TrashNet model has one broader `plastic` class. Any mapping must be documented and only category-compatible crops evaluated. Report the exact held-out subset and its sampling method. Do not treat the unlabeled `all_image_urls.csv` as a ground-truth label file.

### 7. Data improvement plan
After error inspection, refine the strategy without consulting the test split: audit ambiguous paper/cardboard/plastic and minority-class labels, collect/annotate more conveyor examples with permission and license compliance, evaluate augmentations that model occlusion and clutter, and measure cross-domain macro-F1. No improvement is claimed without evidence.

### 8. Reproducibility
Seeds: {val(conf,'seed')}; batch size: {val(conf,'batch_size')}; image size: {exp_size}; device: {val(conf,'device')}. Sources: Colab notebook, `config.json`, `split_manifest.csv`, `class_counts.csv`, `requirements.txt`, three run histories, model checkpoints, the audit record, and `test_metrics.json`.

**Checklist generated from output folder:**
{checks}

### 9. Phase 2 plan (15 November 2026)
M4: fine-tune a ZeroWaste-f detector and assess per-class mAP50/mAP50-95, confidence/NMS sweep, false positives/negatives and transfer to TACO. M5: fine-tune instance segmentation and evaluate mask mAP, IoU and Dice, using mask centroids as pick points. M8: end-to-end ONNX export and numerical parity, HTTP service, p50/p95/p99 latency, throughput, memory and a go/no-go decision defined in advance. The fifth team member's advanced deliverable must be approved with the instructor and is not automatically bonus work.

### References
1. KAUST Academy. AI251P Applied Computer Vision: Capstone Project Guidelines and Example Topics, October 2026, sections 4–6 and 8.9.
2. Thung, G., and Yang, M. TrashNet. https://github.com/garythung/trashnet .
3. ZeroWaste dataset: https://github.com/dbash/zerowaste .
4. TACO: https://github.com/pedropro/TACO .
'''
    dest=root/'PHASE1_REPORT_AUTOFILLED.md'
    dest.write_text(content,encoding='utf-8')
    ready=sum(check.values())
    (root/'PROGRESS_STATUS.json').write_text(json.dumps({'generated_at':str(datetime.datetime.now()),'complete_auto_checks':ready,'total_auto_checks':len(check),'checks':check,'note':'3 checks always need team confirmation: W&B online, team rerun, approval'},indent=2),encoding='utf-8')
    print('Report generated:',dest,'; auto checks:',ready,'/',len(check))
    for k,v in check.items():print(('OK  ' if v else 'TODO'),k)
    return dest


In [ ]:
import importlib.metadata as meta
pkgs=['torch','torchvision','numpy','Pillow','pandas','scikit-learn','scipy','matplotlib','seaborn','wandb','imagehash','huggingface-hub','tabulate']
reqs=[]
for x in pkgs:
    try:reqs.append(f'{x}=={meta.version(x)}')
    except meta.PackageNotFoundError:pass
(OUT/'requirements.txt').write_text('\n'.join(reqs)+'\n')
ct=pd.read_csv(OUT/'class_counts.csv',index_col=0)
with open(OUT/'dataset_card.md','w') as f:
    f.write(f"""# Dataset Card — KAUST Project 9 (TrashNet)\n\n- **Source:** https://huggingface.co/datasets/garythung/trashnet, resized archive (`dataset-resized.zip`). Original TrashNet attributed to Thung and Yang.\n- **Purpose:** Material classification of single waste items; not an industrial conveyor benchmark.\n- **License:** MIT as indicated by the dataset repository; follow source license, no rehosting raw photos.\n- **Classes:** {', '.join(CLASSES)}.\n- **Verified after removal:** {len(df)} images; exact file duplicates removed: {len(exact_repeats)}; damaged: {len(bad)}.\n- **Split:** stratified grouped ~60/20/20 via pHash (threshold <= 2 bit). Seed {SEED}. See split_manifest.csv and class_counts.csv for exact counts.\n- **Train/val/test roles:** train optimization; validation for model selection; test once after selection.\n- **Image preprocessing:** RGB, training random crop + flip + brightness/contrast jitter, resize to {IMG_SIZE}, ImageNet normalization for comparability.\n- **Audit:** {audit_text}\n- **Limitations:** white-background single-object images; strong domain gap to cluttered ZeroWaste conveyor; pHash groups reduce but cannot prove absence of scene/source leakage without camera/session metadata; class imbalance. ZeroWaste materials only partially align to six TrashNet classes.\n- **Restrictions:** avoid redistribution of public datasets; retain image ownership and licenses; no people/PII expected but inspect external data.\n\n## Class balance by split\n\n{ct.to_markdown()}\n""")
shutil.copy2(OUT/'checkpoints'/f'{best_kind}_best.pt',OUT/'best_model.pt')
(OUT/'submission_README.txt').write_text('KAUST AI251P Project 9 — Phase 1\n1. Run KAUST_Project9_Phase1_Colab.ipynb to generate these outputs.\n2. Verify label_audit_review.csv is HUMAN-REVIEWED; rerun audit summary + dataset-card cells.\n3. Upload figures, config, split_manifest.csv, results and best_model.pt.\n4. Include W&B URL or synchronize offline runs; show all three experiments.\n5. If ZeroWaste cross-domain test remains pending, acknowledge omission instead of inventing numbers.\n6. No raw image data should be shared in your repository.\n')
# Generate an evidence-backed report and progress status AFTER the last output exists.
main(str(OUT))
archive=Path('/content/KAUST_Project9_Phase1_Results.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and 'checkpoints' not in p.parts and 'wandb' not in p.parts:
            z.write(p,arcname=p.relative_to(OUT))
print('Saved to:',archive,'\nAll outputs:',OUT)
print('Selected best model:',best_kind,'| audit:',audit_text)
print('W&B offline? Sync your saved offline runs to the online project before submitting.')
print('AUTO REPORT:',OUT/'PHASE1_REPORT_AUTOFILLED.md')
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass

### Before submission
- Check that training has finished and model checkpoints exist.
- Review the label-audit sample.
- Verify the W&B run links.
- Explain at least three mistakes in the report using actual image examples.
- Confirm each team member can explain their contribution.
- Do not report results for tests that were not run.
